# Set Configuration

In [1]:
from diffusion_hash_inv.config import MainConfig, HashConfig, MessageConfig, OutputConfig, Byte2RGBConfig
from diffusion_hash_inv.main import MainEP
from diffusion_hash_inv.utils import FileIO
from diffusion_hash_inv.main import RuntimeConfig

length = 16
iteration = 2**length

main_cfg = MainConfig(
    verbose_flag=False,
    clean_flag=True,
    debug_flag=False,
    make_image_flag=False,
)
hash_cfg = HashConfig(
    hash_alg="md5",
    length=length,
)
message_cfg = MessageConfig(
    message_flag=False,
    length=length,
    random_flag=False,
    seed_flag=True,
)
output_cfg = OutputConfig()
byte2rgb_cfg = Byte2RGBConfig()
runtime_cfg = RuntimeConfig(
    main=main_cfg,
    message=message_cfg,
    hash=hash_cfg,
    output=output_cfg,
    rgb=byte2rgb_cfg,
)


io_controller = FileIO(main_config=main_cfg, output_cfg=output_cfg)

Clearing generated files...



In [2]:
main = MainEP(runtime_config=runtime_cfg, file_controller=io_controller)
main.run(mode="sequential")

Main Entry Point Initialized.
Program Start Time: 2026-04-27 12:30:34.604209+09:00
Hash Algorithm: MD5
Message Length: 16
Data Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/data
Output Directory: /Users/choisoonwook/Experiments_local/Diffusion_HASH_inverse/output
Running in sequential mode.
Iteration count set to: 65536


Hash Generation Progress: 100%|██████████| 65536/65536 [00:58<00:00, 1116.88iteration/s, Hash Algorithm=MD5, Message Length=16]

Hash Calculation time: 58 s, 688 ms, 756 us, 42 ns
Process completed.

Total Execution Time: 58 s, 688 ms, 756 us, 42 ns



In [ ]:
from diffusion_hash_inv.logger.logger import Logs

logs = Logs.get_logs(io_controller, hash_cfg, main_cfg)
print(len(logs))
print(logs[0])

In [ ]:
_logs = io_controller.get_latest_files_by_date(hash_cfg.hash_alg, hash_cfg.length)
print(_logs[0])

In [ ]:
def get_step4(logs):
    step4_logs = []
    for log in logs:
        _tmp = list(log.values())
        assert len(_tmp) == 1, "Each log dictionary should contain exactly one key-value pair."
        log_dict = list(log.values())[0]
        if "Logs" in log_dict and "4th Step" in log_dict["Logs"]:
            step4_logs.append(log_dict["Logs"]["4th Step"])
    return step4_logs

In [ ]:
from typing import List, Dict, Any

step4_logs: List[Dict[str, Any]] = get_step4(logs)
print(step4_logs[0])
beta_schedule = list()

In [ ]:
def cumulative_block(byte_list: List[bytes], block: bytes, indent: int = 0) -> List[bytes]:
    """
    Seperate Block into bytes and cumulatively add to byte_list. Return the updated byte_list.
    """
    _byte = 0
    for byte in block:
        _byte = byte if len(byte_list) == 0 else byte + byte_list[-1]
        # _byte = _byte % (0xff + 1) # Ensure byte value stays within 0-255
        if main_cfg.verbose_flag:
            # print(f"{'\t' * indent}Byte: {byte}")
            # print(f"{'\t' * (indent+1)}Cumulative Byte: {_byte}")
            pass
        byte_list.append(_byte)
    return byte_list


def make_beta_schedule(step4_logs: Dict[str, Any]) -> List[float]:
    step4_log: Dict[str, Any] = list(step4_logs.values())
    beta_schedule = []
    for log in step4_log:
        for key, value in log.items():
            if main_cfg.verbose_flag:
                # print(f"Key: {key}")
                pass
            for k, v in value.items():
                _v = Logs.str_to_bytes(v)
                if main_cfg.verbose_flag:
                    # print(f"\tKey: {k}, Value: {v}")
                    # print(f"\t\tConverted Value: {_v}")
                    pass
                cumulative_block(beta_schedule, _v, indent=3)
                
    return beta_schedule

In [ ]:
from tqdm import tqdm

step4_log_process = tqdm(step4_logs, desc="Processing Step 4 Logs")
for log in step4_log_process:
    beta_schedule.append(make_beta_schedule(log))

In [ ]:
print(len(beta_schedule))
print(len(beta_schedule[0]))
print(beta_schedule[0])
print(beta_schedule[1])

In [ ]:
if main_cfg.verbose_flag:
    width = len(str(iteration))
    for i, beta in enumerate(beta_schedule):
        print(f"Beta Schedule {i:0{width}}:")
        print(f"{'\t' * 1}Beta: ", end="")
        for j, b in enumerate(beta):
            print(f"{b:03},", end=" ")
        print("\n")

In [ ]:
import numpy as np

beta_array = np.array(beta_schedule)
print(beta_array.shape)
min = np.min(beta_array, axis=0)
max = np.max(beta_array, axis=0)
mean = np.mean(beta_array, axis=0)
var = np.var(beta_array, axis=0)
std = np.std(beta_array, axis=0)

np.set_printoptions(threshold=np.inf, linewidth=np.inf)

print(f"Min: {min}, Length: {len(min)}")
print(f"Max: {max}, Length: {len(max)}")
print(f"Mean: {mean}, Length: {len(mean)}")
print(f"Variance: {var}, Length: {len(var)}")
print(f"Standard Deviation: {std}, Length: {len(std)}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plt.plot(min, label='Min', color='blue')
plt.legend()
plt.title('Boxplot of Beta Schedule Statistics')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(max, label='Max', color='orange')
plt.legend()
plt.title('Boxplot of Beta Schedule Statistics')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(mean, label='Mean', color='green')
plt.legend()
plt.title('Boxplot of Beta Schedule Statistics')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(var, label='Variance', color='red')
plt.legend()
plt.title('Boxplot of Beta Schedule Statistics')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(std, label='Standard Deviation', color='purple')
plt.legend()
plt.title('Boxplot of Beta Schedule Statistics')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
import mlx.core as mx
betas = mx.linspace(start=1e-4, stop=2e-2, num=len(beta_schedule[0]))
betas = np.array(betas, dtype=np.float64)
print(betas)
print(beta_schedule[0])
candidate_betas = np.multiply(beta_schedule[0], betas)
candidate_betas = np.array(candidate_betas, dtype=np.float64)
print(candidate_betas)

str_max = 0
for i in range(0, len(beta_schedule)):
    str_len = len(str(beta_schedule[i]))
    if str_len > str_max:
        str_max = str_len
print(f"Max String Length: {str_max}")

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_schedule[0], label='Beta Schedule 0', color='red')
plt.legend()
plt.title('Beta Schedule 0')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(betas, label='Betas', color='blue')
plt.legend()
plt.title('Candidate Betas')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(candidate_betas, label='Candidate Betas', color='cyan')
plt.legend()
plt.title('Candidate Betas')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()